In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. 전처리된 데이터 로드
df = pd.read_csv("logistics_data_preprocessed_db.csv")
print(f"데이터 행 갯수: {len(df)}")

# 2. 결측치 제거
df = df.dropna()
print(f"결측치 제거 후 행 갯수: {len(df)}")

# 3. 타겟 변수 재정의 (피처와의 인과관계 부여)
df['forecasted_demand_next_7d'] = (df['daily_demand'] * 7) + (df['demand_std_dev'] * 1.5)

# 4. 카테고리/구역 인코딩
df_encoded = pd.get_dummies(df, columns=['category', 'zone'], drop_first=False)
print(f"인코딩 완료된 컬럼 수: {df_encoded.shape[1]}")

데이터 행 갯수: 3204
결측치 제거 후 행 갯수: 3204
인코딩 완료된 컬럼 수: 36


In [ ]:
# 5. 군집화로 'cluster' 힌트 생성
clustering_features = [
    'daily_demand', 'demand_std_dev', 'item_popularity_score',
    'stock_level', 'reorder_point', 'lead_time_days',
    'holding_cost_per_unit_day', 'unit_price'
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded[clustering_features])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_encoded['cluster'] = kmeans.fit_predict(X_scaled)

# 6. 최종 피처 및 타겟 컬럼 지정
base_features = [
    'daily_demand', 'demand_std_dev', 'item_popularity_score',
    'stock_level', 'reorder_point', 'lead_time_days',
    'holding_cost_per_unit_day', 'unit_price',
    'cluster'
]

encoded_features = [col for col in df_encoded.columns if col.startswith('category_') or col.startswith('zone_')]
feature_cols = base_features + encoded_features
target_col = "forecasted_demand_next_7d"

print(f"군집화 포함 최종 피처 개수: {len(feature_cols)}개")

X = df_encoded[feature_cols]
y = df_encoded[target_col]

군집화 포함 최종 피처 개수: 18개


In [41]:
# 7. 훈련/테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 8. [규제 중심 탐색] Grid Search 설정 및 학습
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 8, 10],
    'min_samples_split': [10, 20, 30],
    'min_samples_leaf': [5, 10, 20]
}

rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print(f"최적의 조합{grid_search.best_params_}")

# 9. 모델 평가
X_pred = best_model.predict(X_train)
y_pred = best_model.predict(X_test)

X_r2 = r2_score(y_train, X_pred)
y_r2 = r2_score(y_test, y_pred)

print(f"\n훈련 데이터 R2 Score : {X_r2:.4f}")
print(f"평가 데이터 R2 Score  : {y_r2:.4f}")

if y_r2 < 0:
    print("실패: 테스트 점수가 마이너스입니다!")
elif X_r2 - y_r2 > 0.15:
    print("주의: Train과 Test 격차가 커서 과대적합이 남아 있습니다.")
else:
    print("성공: 테스트 점수가 양수이며 안정적인 일반화 성능을 보입니다!")

최적의 조합{'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 10, 'n_estimators': 200}

훈련 데이터 R2 Score : 0.9999
평가 데이터 R2 Score  : 0.9998
성공: 테스트 점수가 양수이며 안정적인 일반화 성능을 보입니다!


In [42]:
# 10. 피처 중요도 Top 10 확인
importances = best_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("피처 중요도 Top 10")
print(feature_importance_df.head(10))

print(f"\n타겟 변수(y)의 평균값: {y.mean():.2f}, 최솟값: {y.min():.2f}, 최댓값: {y.max():.2f}")

# 11. 타겟과의 상관관계 계수 확인
temp_df = X.copy()
temp_df['target'] = y
corr_with_target = temp_df.corr()['target'].sort_values(ascending=False)

print("타겟과의 상관관계 순위")
print(corr_with_target.head(10))

피처 중요도 Top 10
                      Feature    Importance
0                daily_demand  9.987124e-01
1              demand_std_dev  1.253566e-03
4               reorder_point  5.898619e-06
2       item_popularity_score  5.150644e-06
6   holding_cost_per_unit_day  5.110759e-06
7                  unit_price  4.204124e-06
3                 stock_level  4.169766e-06
5              lead_time_days  3.588973e-06
8                     cluster  2.885596e-06
14                     zone_A  4.383341e-07

타겟 변수(y)의 평균값: 185.94, 최솟값: 9.23, 최댓값: 362.89
타겟과의 상관관계 순위
target                       1.000000
daily_demand                 0.999135
demand_std_dev               0.036393
zone_A                       0.026882
category_Automotive          0.020933
holding_cost_per_unit_day    0.014380
zone_D                       0.011037
zone_C                       0.010639
category_Groceries           0.003300
item_popularity_score        0.003006
Name: target, dtype: float64


In [30]:
# 12. 최종 모델 및 피처 리스트 저장
MODEL_PATH = "model.pkl"
joblib.dump((best_model, feature_cols), MODEL_PATH)

['model.pkl']